<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/05-learning-objectives-optimization.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Learning Objectives and Optimization**

Training turns a modeling assumption into a numerical problem. A parameterized model $f_\theta$ produces predictions, an **objective** assigns a scalar cost to the parameters $\theta$, and an **optimizer** chooses a sequence $\theta_0,\theta_1,\ldots$ intended to reduce that cost.

A common objective has the form

$$
J_S(\theta)
=\frac{1}{n}\sum_{i=1}^{n}\ell(f_\theta(x_i),y_i)
+\lambda\Omega(\theta),
$$

where $S=\{(x_i,y_i)\}_{i=1}^n$ is the training sample, $\ell$ is a per-example loss, $\Omega$ is an optional penalty, and $\lambda$ controls its strength. This formula contains two design decisions that should not be confused:

- the **learning objective** says which fitted solution is preferred;
- the **optimization algorithm** says how the system searches for that solution.

A poor objective can be optimized perfectly and still produce the wrong behavior. A suitable objective can also perform poorly because optimization is unstable, stopped too early, or trapped in an unsuitable region. Chapter 04 explained why empirical fit may or may not generalize; this chapter treats the computational machinery used to obtain that fit. Chapter 06 will return to held-out evaluation, uncertainty, and experimental model selection.

### **Evaluation Metrics versus Training Objectives**

An **evaluation metric** represents the decision quality that matters after training. Accuracy, F1, recall at a fixed precision, ranking utility, expected monetary cost, latency, and subgroup constraints are examples. A **training objective** must additionally support efficient credit assignment: when parameters change slightly, the optimizer needs information about whether the change helped and in which direction.

Many useful metrics are unsuitable for direct gradient optimization:

- zero-one accuracy is constant until a prediction crosses a decision boundary, so its derivative is zero almost everywhere and undefined at the boundary;
- F1 depends jointly on all examples through counts of true and false positives rather than decomposing into independent smooth terms;
- AUC depends on pairs of positive and negative examples;
- a deployment utility may be delayed, censored, constrained, or observed only after a human decision.

A **surrogate loss** replaces the metric during fitting. Logistic loss supplies smooth gradients for classification, hinge loss targets a margin, pairwise losses approximate ranking order, and differentiable cost-weighted losses can reflect asymmetric mistakes. The original metric is still used on validation and test data.

![A training objective is a solver-compatible proxy for a decision metric, and their alignment must be checked on untouched data.](assets/metric-objective-surrogate.svg){fig-align="center" width="100%" fig-alt="Flow from a deployment metric through a surrogate training objective to learned parameters and held-out evaluation"}

Surrogate design requires **alignment**. A lower cross-entropy generally encourages better conditional probabilities, but it does not automatically select the best threshold for a particular false-negative cost. Optimizing average loss can also hide subgroup failures. Some surrogates are statistically consistent for a target metric under assumptions, yet finite samples, model misspecification, regularization, and optimization error can still break practical alignment.

<details>
<summary><strong>Python example: identical accuracy can hide very different training losses</strong></summary>

```python
import numpy as np

y = np.array([1, 0, 1, 1, 0, 0])

# Both probability vectors produce exactly the same labels at threshold 0.5.
confident_probabilities = np.array([0.92, 0.08, 0.85, 0.76, 0.12, 0.20])
barely_correct_probabilities = np.array([0.51, 0.49, 0.51, 0.51, 0.49, 0.49])

def accuracy(labels, probabilities):
    return np.mean((probabilities >= 0.5).astype(int) == labels)

def binary_cross_entropy(labels, probabilities):
    probabilities = np.clip(probabilities, 1e-12, 1 - 1e-12)
    return -np.mean(
        labels * np.log(probabilities)
        + (1 - labels) * np.log(1 - probabilities)
    )

for name, probabilities in [
    ("confident", confident_probabilities),
    ("barely correct", barely_correct_probabilities),
]:
    print(
        f"{name:14s}",
        "accuracy =", round(accuracy(y, probabilities), 3),
        "cross-entropy =", round(binary_cross_entropy(y, probabilities), 3),
    )
```

</details>

Accuracy gives no preference between these models, while cross-entropy rewards probability assigned to the observed class. That richer signal is useful for training, but probability calibration and decision-threshold selection remain separate evaluation questions.


### **Likelihood-Based Learning**

A probabilistic model specifies how observed data would be generated for each parameter value. **Likelihood** reverses the viewpoint: once data $D$ are observed, it treats

$$
L(\theta;D)=p(D\mid\theta)
$$

as a function of $\theta$. It is not a probability distribution over $\theta$ unless it is combined with a prior and normalized. The notation after the semicolon emphasizes that $D$ is fixed while candidate parameters vary.

![Maximum likelihood selects parameters that make the data plausible; negative log-likelihood converts products into sums, while MAP also incorporates a prior.](assets/mle-map-nll.svg){fig-align="center" width="100%" fig-alt="Data flows to a likelihood and maximum likelihood estimate; likelihood plus prior produces a posterior and MAP estimate"}

*Concepts follow [Cornell CS4780, Estimating Probabilities from Data](https://www.cs.cornell.edu/courses/cs5780/2024sp/lectures/lecturenote04.html) and [MIT OpenCourseWare, Maximum Likelihood Estimation](https://ocw.mit.edu/courses/18-650-statistics-for-applications-fall-2016/resources/lecture-3-maximum-likelihood-estimation/).*

#### **Maximum Likelihood Estimation**

The **maximum likelihood estimate (MLE)** is

$$
\widehat\theta_{\mathrm{MLE}}
\in\arg\max_\theta p(D\mid\theta).
$$

If observations are conditionally independent given $\theta$,

$$
p(D\mid\theta)=\prod_{i=1}^{n}p(z_i\mid\theta).
$$

This factorization is an assumption about the data-generating process. Duplicated users, temporally dependent measurements, or grouped observations can make the product likelihood overstate the amount of independent evidence.

In supervised discriminative learning, $z_i=(x_i,y_i)$ is often replaced by conditional likelihood $p_\theta(y_i\mid x_i)$. Inputs are treated as given, and parameters are selected to explain labels conditional on those inputs. In a generative model, the likelihood may instead model $p_\theta(x_i,y_i)$ or $p_\theta(x_i)$.

For Bernoulli observations $y_i\in\{0,1\}$ with success probability $\theta$,

$$
p(D\mid\theta)
=\prod_{i=1}^{n}\theta^{y_i}(1-\theta)^{1-y_i}
=\theta^{n_1}(1-\theta)^{n-n_1},
$$

where $n_1=\sum_i y_i$. Differentiating the log-likelihood gives $\widehat\theta_{\mathrm{MLE}}=n_1/n$, the observed success fraction.

<details>
<summary><strong>Python example: recover a Bernoulli MLE from a likelihood profile</strong></summary>

```python
import numpy as np

observations = np.array([1, 1, 0, 1, 0, 1, 1, 1, 0, 1])
number_of_successes = observations.sum()
sample_size = len(observations)

grid = np.linspace(0.001, 0.999, 5_000)
log_likelihood = (
    number_of_successes * np.log(grid)
    + (sample_size - number_of_successes) * np.log(1 - grid)
)
grid_mle = grid[np.argmax(log_likelihood)]
analytic_mle = number_of_successes / sample_size

print("successes / n:", f"{number_of_successes}/{sample_size}")
print("grid MLE:      ", round(grid_mle, 4))
print("analytic MLE:  ", round(analytic_mle, 4))
```

</details>

MLE is an estimator, not a guarantee of a correct model. If the likelihood family is misspecified, the estimate chooses the closest available explanation under the implied divergence, not the true process.

#### **Maximum a Posteriori Estimation**

Bayes' rule gives

$$
p(\theta\mid D)
=\frac{p(D\mid\theta)p(\theta)}{p(D)}.
$$

The **maximum a posteriori (MAP)** estimate selects the posterior mode:

$$
\widehat\theta_{\mathrm{MAP}}
\in\arg\max_\theta p(\theta\mid D)
=\arg\max_\theta\left[\log p(D\mid\theta)+\log p(\theta)\right],
$$

because the evidence $p(D)$ does not depend on $\theta$. MAP differs from full Bayesian prediction, which integrates over parameter uncertainty rather than replacing the posterior by one mode.

A prior acts like a parameter preference. For a Gaussian prior $p(\theta)\propto\exp(-\|\theta\|_2^2/(2\tau^2))$, negative log-prior contributes an L2 penalty. A Laplace prior contributes an L1 penalty. This interpretation depends on parameterization and scaling: a prior that is simple in one coordinate system need not remain simple after transformation.

For a Bernoulli parameter with $\theta\sim\operatorname{Beta}(\alpha,\beta)$, the interior MAP estimate is

$$
\widehat\theta_{\mathrm{MAP}}
=\frac{n_1+\alpha-1}{n+\alpha+\beta-2}.
$$

The formula requires $\alpha,\beta>1$ for an interior mode. Boundary cases need separate treatment.

<details>
<summary><strong>Python example: observe how a Beta prior influences MAP at different sample sizes</strong></summary>

```python
def bernoulli_mle_map(successes, trials, alpha=2.0, beta=2.0):
    mle = successes / trials
    map_estimate = (successes + alpha - 1) / (trials + alpha + beta - 2)
    return mle, map_estimate

for successes, trials in [(1, 3), (7, 10), (700, 1_000)]:
    mle, map_estimate = bernoulli_mle_map(successes, trials)
    print(
        f"{successes:4d}/{trials:<4d}",
        f"MLE={mle:.4f}",
        f"MAP Beta(2,2)={map_estimate:.4f}",
    )
```

</details>

The prior has visible influence when data are scarce and becomes comparatively small as $n$ grows. A confident but wrong prior can harm estimation, so prior choice should encode defensible information rather than being treated as free regularization.

#### **Negative Log-Likelihood**

Because logarithm is strictly increasing,

$$
\arg\max_\theta p(D\mid\theta)
=\arg\max_\theta\log p(D\mid\theta)
=\arg\min_\theta\left[-\log p(D\mid\theta)\right].
$$

Under conditional independence,

$$
-\log p(D\mid\theta)
=-\sum_{i=1}^{n}\log p(z_i\mid\theta).
$$

The **negative log-likelihood (NLL)** converts products of tiny probabilities into sums, improves numerical stability, and naturally decomposes across examples and mini-batches. Averaging rather than summing NLL does not change an unregularized minimizer, but it changes gradient scale and the relative meaning of a fixed regularization coefficient.

Common losses are NLLs under specific observation models:

| Observation model | Negative log-likelihood, up to constants | Familiar loss |
|---|---|---|
| Gaussian with fixed variance | squared residual | mean squared error |
| Laplace with fixed scale | absolute residual | mean absolute error |
| Bernoulli | binary cross-entropy | logistic loss |
| Categorical | negative log predicted class probability | multiclass cross-entropy |

<details>
<summary><strong>Python example: log space prevents likelihood underflow</strong></summary>

```python
import numpy as np

# A long sequence of individually plausible but small probabilities.
probabilities = np.full(1_000, 0.01)

direct_product = np.prod(probabilities)
log_likelihood = np.sum(np.log(probabilities))
negative_log_likelihood = -log_likelihood

print("direct probability product:", direct_product)
print("log-likelihood:            ", round(log_likelihood, 3))
print("negative log-likelihood:   ", round(negative_log_likelihood, 3))
print("finite in log space:       ", np.isfinite(negative_log_likelihood))
```

</details>

Stable software usually goes further by combining transformations. For example, `log_softmax` computes log probabilities without first materializing potentially overflowing exponentials, as introduced in Chapter 03.


### **Loss Functions**

A loss is not merely a convenient curve. It determines which errors receive large gradients, what statistical quantity the optimal prediction estimates, whether outliers dominate, and which optimization methods are valid. A good choice follows the target, noise process, action costs, and robustness requirements.

#### **Regression Losses**

For residual $r=y-\widehat y$, common regression losses include

$$
\ell_{\mathrm{square}}(r)=\frac{1}{2}r^2,
\qquad
\ell_{\mathrm{absolute}}(r)=|r|,
$$

and Huber loss with transition $\delta>0$,

$$
\ell_\delta(r)=
\begin{cases}
\frac{1}{2}r^2,&|r|\leq\delta,\\
\delta\left(|r|-\frac{1}{2}\delta\right),&|r|>\delta.
\end{cases}
$$

Squared loss has gradient proportional to $r$, so large residuals exert rapidly increasing influence. Absolute loss has bounded subgradient $\operatorname{sign}(r)$ away from zero and targets a conditional median rather than a conditional mean. Huber is quadratic near zero and linear in the tails, combining smooth local fitting with bounded tail influence.

For quantile level $\tau\in(0,1)$, pinball loss

$$
\rho_\tau(r)=
\begin{cases}
\tau r,&r\geq0,\\
(\tau-1)r,&r<0
\end{cases}
$$

targets the conditional $\tau$-quantile and supports asymmetric prediction intervals or unequal under- and over-prediction costs.

<details>
<summary><strong>Python example: compare how regression losses weight the same residuals</strong></summary>

```python
import numpy as np

residuals = np.array([-8.0, -2.0, -0.5, 0.2, 1.5, 6.0])
delta = 1.0
quantile = 0.8

squared = 0.5 * residuals ** 2
absolute = np.abs(residuals)
huber = np.where(
    np.abs(residuals) <= delta,
    0.5 * residuals ** 2,
    delta * (np.abs(residuals) - 0.5 * delta),
)
pinball = np.where(
    residuals >= 0,
    quantile * residuals,
    (quantile - 1.0) * residuals,
)

print("residual | squared | absolute | Huber(delta=1) | pinball(tau=0.8)")
for values in zip(residuals, squared, absolute, huber, pinball):
    print(f"{values[0]:8.1f} | {values[1]:7.2f} | {values[2]:8.2f} | {values[3]:14.2f} | {values[4]:16.2f}")
```

</details>

The asymmetric pinball values are not an error: positive residuals mean the model underpredicted $y$ under this residual convention and receive weight $\tau=0.8$.

#### **Classification Losses**

For binary labels $y\in\{-1,+1\}$ and score $s=f_\theta(x)$, the signed **margin** is $m=ys$. Correct confident predictions have large positive margin; incorrect predictions have negative margin. Zero-one loss $\mathbf1[m\leq0]$ matches accuracy but provides no useful gradient.

![Convex surrogate losses replace discontinuous zero-one loss with different smoothness, margin, and robustness properties.](assets/classification-surrogate-losses.png){fig-align="center" width="78%" fig-alt="Scikit-learn plot comparing zero-one, hinge, perceptron, log, squared hinge, and modified Huber classification losses"}

*Source: [scikit-learn, SGD: Convex Loss Functions](https://scikit-learn.org/stable/auto_examples/linear_model/plot_sgd_loss_functions.html), BSD-3-Clause.*

Important surrogates are

$$
\ell_{\mathrm{logistic}}(m)=\log(1+e^{-m}),
\qquad
\ell_{\mathrm{hinge}}(m)=\max(0,1-m).
$$

Logistic loss is smooth and has a probabilistic interpretation. Its derivative with respect to margin is $-1/(1+e^m)$, so severely wrong examples receive a gradient near $-1$. Hinge loss is zero once margin reaches 1 and has a subgradient at its corner. Squared hinge penalizes margin violations more strongly but becomes more sensitive to extreme errors.

For multiclass probabilities $p_k$ and true class $c$, cross-entropy is $-\log p_c$. Label smoothing, focal loss, and class weights modify gradient allocation, but each changes the implied objective and may affect probability calibration.

<details>
<summary><strong>Python example: inspect classification losses and margin gradients</strong></summary>

```python
import numpy as np

margins = np.array([-3.0, -1.0, 0.0, 0.5, 1.0, 3.0])
zero_one = (margins <= 0).astype(float)
hinge = np.maximum(0.0, 1.0 - margins)
logistic = np.logaddexp(0.0, -margins)  # stable log(1 + exp(-margin))
logistic_gradient = -1.0 / (1.0 + np.exp(margins))

print("margin | zero-one | hinge | logistic | d(logistic)/d(margin)")
for values in zip(margins, zero_one, hinge, logistic, logistic_gradient):
    print(f"{values[0]:6.1f} | {values[1]:8.1f} | {values[2]:5.2f} | {values[3]:8.4f} | {values[4]:23.4f}")
```

</details>

#### **Margin and Ranking Losses**

Margin losses care about separation, not only the predicted class. Hinge loss requires $ys\geq1$ before an example stops contributing, which supports the maximum-margin interpretation of support vector machines.

Ranking objectives compare items. For a preferred item $i$ and less relevant item $j$, define score difference $d=s_i-s_j$. A pairwise logistic loss is

$$
\ell_{\mathrm{rank}}(d)=\log(1+e^{-d}),
$$

which decreases as the preferred item outranks the other by a larger margin. Pairwise hinge uses $\max(0,1-d)$. Triplet losses similarly compare an anchor with positive and negative examples in representation learning.

Pairwise objectives can require $O(n^2)$ comparisons, and sampled negatives change the effective training distribution. Optimizing pairwise accuracy also need not optimize top-$k$ utility or position-discounted ranking metrics, so sampling and weighting should reflect deployment.

<details>
<summary><strong>Python example: optimize one pairwise logistic ranking constraint</strong></summary>

```python
import numpy as np

preferred_score = 0.30
other_score = 0.80
learning_rate = 0.4

print("step | preferred | other | score difference | pairwise loss")
for step in range(6):
    difference = preferred_score - other_score
    loss = np.logaddexp(0.0, -difference)
    print(f"{step:4d} | {preferred_score:9.4f} | {other_score:5.4f} | {difference:16.4f} | {loss:13.4f}")

    # d loss / d difference = -sigmoid(-difference).
    gradient_difference = -1.0 / (1.0 + np.exp(difference))
    preferred_score -= learning_rate * gradient_difference
    other_score += learning_rate * gradient_difference
```

</details>

The two scores move in opposite directions because their difference is the object being optimized. Real ranking models share parameters across many items, so one update affects many comparisons simultaneously.

#### **Robust Losses**

A loss is **robust** when a small fraction of extreme observations cannot arbitrarily dominate the fitted result under the assumed contamination model. Huber loss limits residual influence but remains convex. Redescending losses such as Tukey's biweight can eventually give zero influence to extreme residuals, but they are non-convex and may introduce local optima.

![Huber regression remains closer to the central trend than squared-loss ridge regression when strong outliers are present.](assets/huber-vs-ridge.png){fig-align="center" width="78%" fig-alt="Scikit-learn comparison of Huber regression at several epsilon values against ridge regression in the presence of outliers"}

*Source: [scikit-learn, HuberRegressor vs Ridge on a Dataset with Strong Outliers](https://scikit-learn.org/stable/auto_examples/linear_model/plot_huber_vs_ridge.html), BSD-3-Clause.*

Robustness is not a substitute for investigating data errors. If rare extreme cases are legitimate and important, downweighting them can be harmful. Robust loss addresses a specified error process; it does not certify robustness to covariate shift, adversarial inputs, or mislabeled subgroups.

<details>
<summary><strong>Python example: compare squared and Huber fits after label contamination</strong></summary>

```python
import numpy as np
from sklearn.linear_model import HuberRegressor, LinearRegression
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(17)
x_train = rng.uniform(-2.0, 2.0, 90)[:, None]
y_clean = 1.5 + 2.2 * x_train[:, 0] + rng.normal(scale=0.35, size=len(x_train))
y_contaminated = y_clean.copy()

outlier_indices = rng.choice(len(y_contaminated), size=10, replace=False)
y_contaminated[outlier_indices] += rng.normal(loc=0.0, scale=12.0, size=10)

x_test = rng.uniform(-2.0, 2.0, 5_000)[:, None]
y_test = 1.5 + 2.2 * x_test[:, 0] + rng.normal(scale=0.35, size=len(x_test))

models = {
    "squared loss": LinearRegression(),
    "Huber loss": HuberRegressor(epsilon=1.35, alpha=0.0),
}

for name, model in models.items():
    model.fit(x_train, y_contaminated)
    test_mse = mean_squared_error(y_test, model.predict(x_test))
    print(
        f"{name:12s}",
        "intercept =", round(float(model.intercept_), 3),
        "slope =", round(float(model.coef_[0]), 3),
        "clean test MSE =", round(test_mse, 4),
    )
```

</details>


### **Regularized Objectives**

Chapter 04 introduced regularization as capacity control. Here the focus is its objective geometry and computational consequence. For linear parameters $w$, elastic-net regularization can be written

$$
J(w)=\widehat R_S(w)
+\lambda\left[
\alpha\|w\|_1+\frac{1-\alpha}{2}\|w\|_2^2
\right],
$$

with $\lambda\geq0$ and mixing parameter $\alpha\in[0,1]$. Setting $\alpha=1$ gives L1; $\alpha=0$ gives L2 under this convention.

#### **L1, L2, and Elastic-Net Penalties**

L2 is smooth, with gradient $\nabla_w\frac{1}{2}\|w\|_2^2=w$. It can be included directly in gradient, Newton, and quasi-Newton methods. L1 is not differentiable at zero. Using an arbitrary derivative of zero at that point would destroy its sparsity mechanism; coordinate descent, subgradient methods, or proximal algorithms handle the corner correctly.

Elastic net combines L1 sparsity with L2 stabilization. When predictors are strongly correlated, pure Lasso may select one unstable member of a group, whereas the L2 component encourages grouped shrinkage. All coefficient penalties depend on feature scale, so preprocessing must be fitted on training data and included in the pipeline.

<details>
<summary><strong>Python example: compare coefficient structures induced by L1, L2, and elastic net</strong></summary>

```python
import numpy as np
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(29)
n_train, n_test, n_features = 130, 3_000, 20

# Correlated features are generated from four latent factors.
mixing = rng.normal(size=(4, n_features))
x_train = rng.normal(size=(n_train, 4)) @ mixing + 0.2 * rng.normal(size=(n_train, n_features))
x_test = rng.normal(size=(n_test, 4)) @ mixing + 0.2 * rng.normal(size=(n_test, n_features))
true_weights = np.zeros(n_features)
true_weights[[1, 6, 13, 18]] = [2.0, -1.6, 1.2, -0.9]
y_train = x_train @ true_weights + rng.normal(scale=0.8, size=n_train)
y_test = x_test @ true_weights + rng.normal(scale=0.8, size=n_test)

estimators = {
    "L2 Ridge": Ridge(alpha=0.5),
    "L1 Lasso": Lasso(alpha=0.04, max_iter=20_000),
    "Elastic net": ElasticNet(alpha=0.04, l1_ratio=0.5, max_iter=20_000),
}

print("model       test MSE  coefficient norm  non-zero")
for name, estimator in estimators.items():
    model = make_pipeline(StandardScaler(), estimator)
    model.fit(x_train, y_train)
    coefficients = model[-1].coef_
    print(
        f"{name:11s} {mean_squared_error(y_test, model.predict(x_test)):8.4f}"
        f" {np.linalg.norm(coefficients):17.4f}"
        f" {np.count_nonzero(np.abs(coefficients) > 1e-8):9d}"
    )
```

</details>

The result depends on the chosen hyperparameters and sample. The point is the structure of the solutions: Ridge remains dense, Lasso can be sparse, and elastic net interpolates between those behaviors.

#### **Constrained and Penalized Forms**

Regularization can be stated as a penalty

$$
\min_w\ \widehat R_S(w)+\lambda\Omega(w)
$$

or as a constraint

$$
\min_w\ \widehat R_S(w)
\quad\text{subject to}\quad
\Omega(w)\leq c.
$$

![A penalty prices complexity while a constraint limits it; L1 and L2 feasible regions produce different contact geometry.](assets/regularization-constraint.svg){fig-align="center" width="100%" fig-alt="Penalized and constrained regularization forms plus L1 diamond and L2 circle geometry"}

For many convex problems, each useful constraint radius $c$ corresponds to at least one multiplier $\lambda$ that gives the same solution. The numerical mapping is problem-dependent and may not be one-to-one. A value of `alpha=1` in one library can also represent a differently normalized objective in another, so compare full definitions rather than parameter names.

The constrained form is natural when a domain supplies a hard budget, such as a maximum norm, memory use, turnover, or fairness disparity. The penalized form is often easier for unconstrained solvers and expresses a trade-off rather than a strict limit.

<details>
<summary><strong>Python example: match an L2 penalty with an equivalent norm constraint</strong></summary>

```python
import numpy as np

# Minimize 0.5 * ||w - a||^2. Its unconstrained optimum is a.
a = np.array([3.0, -1.0])
regularization = 1.5

# Penalized problem: 0.5||w-a||^2 + lambda/2 * ||w||^2.
penalized_solution = a / (1.0 + regularization)

# Choose the constraint radius equal to the norm of that penalized solution.
radius = np.linalg.norm(penalized_solution)
constrained_solution = radius * a / np.linalg.norm(a)

print("penalized solution: ", penalized_solution.round(6))
print("constraint radius:  ", round(radius, 6))
print("constrained solution:", constrained_solution.round(6))
print("maximum difference: ", np.max(np.abs(penalized_solution - constrained_solution)))
```

</details>


### **Convex and Non-Convex Optimization**

Optimization guarantees depend less on whether an objective looks like a bowl in a two-dimensional picture and more on mathematical structure: convexity, smoothness, strong convexity, constraints, stochasticity, and conditioning.

#### **Convex Sets and Convex Functions**

A set $C$ is **convex** if every line segment between points in the set remains in the set:

$$
x,y\in C,\ t\in[0,1]
\quad\Longrightarrow\quad
tx+(1-t)y\in C.
$$

A function $f:C\rightarrow\mathbb R$ is convex if

$$
f(tx+(1-t)y)
\leq tf(x)+(1-t)f(y)
$$

for every $x,y\in C$ and $t\in[0,1]$. Geometrically, the chord between two graph points lies above the graph. For differentiable $f$, an equivalent first-order condition is

$$
f(y)\geq f(x)+\nabla f(x)^\top(y-x),
$$

meaning every tangent hyperplane is a global lower bound.

**Strict convexity** prevents two distinct points from both being minimizers. **Strong convexity** adds a quadratic lower-curvature bound and supports faster rates. Smoothness controls upper curvature through a Lipschitz gradient. A function can be convex but non-smooth, as with $|w|$.

![A convex objective has no suboptimal local minima, whereas a non-convex objective may contain multiple minima, saddles, and flat regions.](assets/convex-nonconvex-landscape.svg){fig-align="center" width="100%" fig-alt="Side-by-side convex bowl and non-convex landscape with local minima and a saddle"}

*Definitions and guarantees follow [Boyd and Vandenberghe, Convex Optimization](https://web.stanford.edu/~boyd/cvxbook/).*

<details>
<summary><strong>Python example: test Jensen's inequality numerically</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(37)
A = rng.normal(size=(8, 3))
b = rng.normal(size=8)

def convex_objective(w):
    residual = A @ w - b
    return 0.5 * residual @ residual

def nonconvex_objective(w):
    return w[0] ** 4 - 3.0 * w[0] ** 2 + 0.2 * w[1] ** 2

def largest_jensen_violation(function, trials=5_000):
    largest = -np.inf
    for _ in range(trials):
        x = rng.normal(size=3)
        y = rng.normal(size=3)
        t = rng.uniform()
        violation = function(t * x + (1 - t) * y) - (
            t * function(x) + (1 - t) * function(y)
        )
        largest = max(largest, violation)
    return largest

print("largest convex-objective violation:   ", f"{largest_jensen_violation(convex_objective):.3e}")
print("largest non-convex-objective violation:", f"{largest_jensen_violation(nonconvex_objective):.3e}")
```

</details>

Random testing cannot prove convexity, but a positive violation disproves it. Proof uses algebraic closure rules, Hessian positive semidefiniteness for twice-differentiable functions, or disciplined convex programming rules.

#### **Local and Global Optima**

A point $\theta^*$ is a **local minimum** if no sufficiently small feasible perturbation lowers the objective. It is **global** if no feasible point anywhere has lower objective. Every local minimum of a convex problem is global. In a strictly convex problem the global minimizer, if it exists, is unique.

For differentiable unconstrained optimization, $\nabla f(\theta)=0$ is a necessary first-order condition for an interior local optimum, but it also holds at maxima and saddles. If the Hessian is positive definite, the stationary point is a strict local minimum; an indefinite Hessian identifies a saddle. A positive semidefinite Hessian alone can be inconclusive when zero-curvature directions exist.

Non-convex does not automatically mean untrainable. Modern overparameterized objectives can contain many connected or similarly performing minima, and saddle points may be more common than harmful isolated local minima. Nevertheless, initialization, optimizer, stochasticity, and parameterization can change which stationary region is reached.

<details>
<summary><strong>Python example: different starts reach different minima of a non-convex objective</strong></summary>

```python
import numpy as np

def objective(w):
    return (w ** 2 - 1.0) ** 2 + 0.10 * w

def gradient(w):
    return 4.0 * w * (w ** 2 - 1.0) + 0.10

def gradient_descent(start, learning_rate=0.05, steps=500):
    w = float(start)
    for _ in range(steps):
        w -= learning_rate * gradient(w)
    return w, objective(w)

for start in [-2.0, -0.2, 0.0, 0.2, 2.0]:
    solution, value = gradient_descent(start)
    print(f"start={start:5.1f} -> solution={solution: .6f}, objective={value:.6f}")
```

</details>

The two minima are not equally good because the linear term tilts the double well. Reporting only that training "converged" would hide which basin was selected.


### **First-Order Optimization**

First-order methods use objective values and gradients but do not explicitly construct the Hessian. They are the default for large machine-learning problems because a gradient can usually be obtained by automatic differentiation at a cost comparable to a small number of forward evaluations. Their practical behavior nevertheless depends on more than the optimizer name: feature scaling, batch construction, learning-rate policy, parameterization, and numerical precision all change the path through parameter space.

The gradient $g_t=\nabla J(\theta_t)$ is the direction of steepest local increase under the Euclidean norm, so $-g_t$ is the direction of steepest local decrease. This is a **local linear statement**:

$$
J(\theta_t+\Delta)\approx J(\theta_t)+g_t^\top\Delta.
$$

It does not promise that an arbitrarily large step along $-g_t$ will reduce the true curved objective. The learning rate determines how far the optimizer trusts this local approximation.

![Batch gradients, noisy mini-batch gradients, momentum, and coordinate-wise adaptive steps follow different paths over the same objective.](assets/gradient-batch-momentum.svg){fig-align="center" width="100%" fig-alt="Comparison of batch gradient descent, stochastic gradients, momentum, and adaptive coordinate-wise steps on an optimization landscape"}

*Conceptual comparison adapted from the trajectory-based explanations in [Why Momentum Really Works](https://distill.pub/2017/momentum/).*

#### **Batch Gradient Descent**

**Batch gradient descent** evaluates the gradient using every training example before each update:

$$
g_t=\nabla J_S(\theta_t)
=\frac{1}{n}\sum_{i=1}^{n}\nabla_\theta\ell_i(\theta_t)
+\lambda\nabla\Omega(\theta_t),
\qquad
\theta_{t+1}=\theta_t-\eta_t g_t.
$$

Here $n$ is the number of training examples, $\ell_i$ is the loss contributed by example $i$, $\eta_t>0$ is the learning rate, and the regularizer is differentiated once rather than once per example. Because the full empirical gradient is deterministic at a fixed $\theta_t$, objective curves are comparatively smooth and stopping tests are meaningful. The cost is one complete pass over the data per update, which can make feedback slow and memory access expensive.

For least squares with

$$
J(w)=\frac{1}{2n}\lVert Xw-y\rVert_2^2,
$$

the gradient is $X^\top(Xw-y)/n$ and the Hessian is $X^\top X/n$. If $L=\lambda_{\max}(X^\top X/n)$, a fixed learning rate no larger than $1/L$ is a conservative choice: it accounts for the direction of greatest curvature. This relationship is why standardization can improve optimization even though it does not add information to the data.

<details>
<summary><strong>Python example: batch gradient descent recovers the least-squares solution</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(7)
n_samples, n_features = 300, 4
X = rng.normal(size=(n_samples, n_features))
true_weights = np.array([2.0, -1.5, 0.0, 0.8])
y = X @ true_weights + rng.normal(scale=0.25, size=n_samples)

def objective(weights):
    residual = X @ weights - y
    return 0.5 * np.mean(residual ** 2)

def gradient(weights):
    return X.T @ (X @ weights - y) / n_samples

# For this quadratic, the largest Hessian eigenvalue controls a stable step.
hessian = X.T @ X / n_samples
lipschitz_constant = np.linalg.eigvalsh(hessian).max()
learning_rate = 1.0 / lipschitz_constant

weights = np.zeros(n_features)
for _ in range(200):
    weights -= learning_rate * gradient(weights)

closed_form_weights = np.linalg.lstsq(X, y, rcond=None)[0]
print("gradient-descent weights:", np.round(weights, 4))
print("least-squares weights:    ", np.round(closed_form_weights, 4))
print("distance between solutions:", f"{np.linalg.norm(weights - closed_form_weights):.2e}")
print("final objective:", f"{objective(weights):.6f}")
```

</details>

Batch gradient descent is attractive when the dataset fits comfortably in memory, gradients are cheap, and deterministic progress matters. It is less attractive when each full pass is costly or when new data arrive continuously. “Batch” describes how the gradient is computed; it does not imply that the full dataset must be materialized in one array, because exact gradients can also be accumulated over several chunks before updating.

#### **Stochastic and Mini-Batch Gradient Descent**

**Stochastic gradient descent (SGD)** updates from one randomly selected example. **Mini-batch gradient descent** uses a subset $B_t$:

$$
\widehat g_t
=\frac{1}{|B_t|}\sum_{i\in B_t}\nabla_\theta\ell_i(\theta_t)
+\lambda\nabla\Omega(\theta_t),
\qquad
\theta_{t+1}=\theta_t-\eta_t\widehat g_t.
$$

If examples are sampled uniformly and the regularizer is handled correctly, $\mathbb E[\widehat g_t\mid\theta_t]=\nabla J_S(\theta_t)$: the mini-batch gradient is an unbiased estimate of the full gradient. Unbiased does not mean exact. Its covariance controls how erratic the update is, and under weak dependence assumptions the variance of an average decreases approximately as $1/|B_t|$.

This noise has two faces. It causes the training objective to fluctuate and prevents a fixed nonzero learning rate from settling exactly at a minimum. It also produces frequent inexpensive updates and can help a non-convex optimizer leave sharp or saddle-like regions. Larger batches improve hardware utilization and gradient accuracy, but after a problem-dependent point they provide diminishing variance reduction relative to their extra computation.

An **epoch** is one nominal pass through the training set. A typical implementation reshuffles indices each epoch, forms non-overlapping mini-batches, computes the mean loss rather than the sum, and updates once per batch. Averaging is important: if a summed loss is used, changing batch size also changes the effective step size. Data with time, user, or group dependence require a sampling policy that respects that structure rather than blind row-level shuffling.

<details>
<summary><strong>Python example: larger mini-batches reduce gradient-estimation noise</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(19)
n_samples, n_features = 512, 6
X = rng.normal(size=(n_samples, n_features))
y = X @ np.array([1.4, -0.8, 0.3, 0.0, 0.6, -1.1]) + rng.normal(
    scale=0.8, size=n_samples
)

weights = np.array([0.2, -0.1, 0.4, 0.3, -0.2, 0.1])

# Per-example gradients for one fixed parameter vector.
residuals = X @ weights - y
per_example_gradients = X * residuals[:, None]
full_gradient = per_example_gradients.mean(axis=0)

for batch_size in [1, 4, 16, 64, 256]:
    squared_errors = []
    for _ in range(500):
        indices = rng.choice(n_samples, size=batch_size, replace=False)
        estimate = per_example_gradients[indices].mean(axis=0)
        squared_errors.append(np.sum((estimate - full_gradient) ** 2))
    print(
        f"batch={batch_size:3d}",
        "mean squared gradient error =",
        f"{np.mean(squared_errors):.5f}",
    )
```

</details>

The trade-off is therefore not “SGD is inaccurate and batches are accurate.” It is a throughput-and-noise design problem. In deep learning, mini-batches dominate because vectorized hardware makes several examples much cheaper than processing them separately. For small convex datasets, full-batch or specialized solvers can reach a high-accuracy solution with fewer tuning decisions.

#### **Momentum and Adaptive Learning Rates**

Plain gradient descent reacts only to the current gradient. **Momentum** carries a velocity that aggregates recent directions:

$$
v_{t+1}=\beta v_t+g_t,
\qquad
\theta_{t+1}=\theta_t-\eta v_{t+1},
$$

where $0\leq\beta<1$ controls memory. Gradients that repeatedly point in the same direction accumulate; components that alternate across a narrow valley partially cancel. This can accelerate progress along shallow directions while damping side-to-side oscillation along steep ones. Some libraries use $(1-\beta)g_t$ in the velocity definition, so learning rates are not directly comparable across conventions. **Nesterov momentum** evaluates or approximates the gradient after looking ahead in the velocity direction, allowing earlier correction.

Adaptive methods assign a separate effective step to each coordinate. Adam maintains exponential moving averages of the gradient and squared gradient:

$$
\begin{aligned}
m_t &= \beta_1m_{t-1}+(1-\beta_1)g_t,\\
v_t &= \beta_2v_{t-1}+(1-\beta_2)g_t\odot g_t,\\
\widehat m_t &= \frac{m_t}{1-\beta_1^t},\qquad
\widehat v_t = \frac{v_t}{1-\beta_2^t},\\
\theta_t &= \theta_{t-1}-\eta
\frac{\widehat m_t}{\sqrt{\widehat v_t}+\epsilon}.
\end{aligned}
$$

$m_t$ estimates a smoothed direction, $v_t$ estimates coordinate-wise gradient scale, $\odot$ denotes element-wise multiplication, bias correction compensates for initialization at zero, and $\epsilon$ prevents division by zero. AdaGrad accumulates squared gradients without forgetting and is useful for sparse features; RMSProp uses exponential forgetting; Adam combines RMS scaling with momentum-like first moments.

Adaptive scaling often makes early training forgiving when gradients differ greatly by coordinate, but it is not a universal guarantee of better final generalization. Its extra state requires roughly two auxiliary numbers per parameter for Adam. Weight decay also needs care: adding an L2 gradient to Adam is not equivalent to shrinking parameters uniformly because the adaptive preconditioner rescales that gradient. AdamW applies decay separately from the loss-gradient update.

<details>
<summary><strong>Python example: GD, momentum, and Adam on an ill-conditioned quadratic</strong></summary>

```python
import numpy as np

# Curvature is 100 times larger in the vertical than horizontal direction.
A = np.diag([1.0, 100.0])
start = np.array([8.0, 8.0])

def objective(point):
    return 0.5 * point @ A @ point

def gradient(point):
    return A @ point

def run_gd(steps=150, learning_rate=0.018):
    point = start.copy()
    for _ in range(steps):
        point -= learning_rate * gradient(point)
    return point

def run_momentum(steps=150, learning_rate=0.010, beta=0.90):
    point = start.copy()
    velocity = np.zeros_like(point)
    for _ in range(steps):
        velocity = beta * velocity + gradient(point)
        point -= learning_rate * velocity
    return point

def run_adam(steps=150, learning_rate=0.15, beta1=0.9, beta2=0.999):
    point = start.copy()
    first_moment = np.zeros_like(point)
    second_moment = np.zeros_like(point)
    for step in range(1, steps + 1):
        grad = gradient(point)
        first_moment = beta1 * first_moment + (1 - beta1) * grad
        second_moment = beta2 * second_moment + (1 - beta2) * grad ** 2
        corrected_first = first_moment / (1 - beta1 ** step)
        corrected_second = second_moment / (1 - beta2 ** step)
        point -= learning_rate * corrected_first / (
            np.sqrt(corrected_second) + 1e-8
        )
    return point

for name, optimizer in [
    ("gradient descent", run_gd),
    ("momentum", run_momentum),
    ("Adam", run_adam),
]:
    solution = optimizer()
    print(
        f"{name:16s}",
        "point =", np.round(solution, 5),
        "objective =", f"{objective(solution):.6e}",
    )
```

</details>

This comparison is illustrative rather than a leaderboard: each method has a different best learning-rate range, and results change with the objective and allotted updates. The right conclusion is that optimizer state changes the geometry of the update. A fair empirical comparison holds the data order and compute budget fixed, tunes each method independently, and reports both progress per update and progress per unit of time.


### **Second-Order and Coordinate Methods**

First-order methods use slope but treat every direction through the same basic update rule. When curvature is available or the objective has exploitable structure, a more specialized method can make far more progress per iteration. The price may be denser linear algebra, additional memory, or restrictions on the form of the objective.

![Newton-type methods rescale by curvature, coordinate descent isolates one variable, and proximal methods separate a smooth term from a structured non-smooth term.](assets/second-order-coordinate-prox.svg){fig-align="center" width="100%" fig-alt="Three-panel comparison of Newton curvature scaling, coordinate descent, and proximal gradient soft thresholding"}

*The proximal interpretation follows [Parikh and Boyd, Proximal Algorithms](https://stanford.edu/~boyd/papers/pdf/prox_algs.pdf).*

#### **Newton and Quasi-Newton Methods**

Newton's method uses a local quadratic rather than a local linear approximation. At $\theta_t$,

$$
J(\theta_t+p)
\approx J(\theta_t)+g_t^\top p+\frac{1}{2}p^\top H_t p,
$$

where $g_t=\nabla J(\theta_t)$ and $H_t=\nabla^2J(\theta_t)$ is the Hessian. Minimizing this quadratic model gives the Newton direction

$$
H_t p_t=-g_t,
\qquad
\theta_{t+1}=\theta_t+\alpha_t p_t.
$$

The linear system should be solved directly; explicitly forming $H_t^{-1}$ is slower and less numerically stable. For a positive-definite Hessian, $p_t$ is a descent direction. Near a well-behaved optimum, full Newton steps can converge quadratically: roughly speaking, the number of correct digits can double at each iteration. Far from the optimum, or when $H_t$ is singular or indefinite, the raw direction can be unreliable. Line search, trust regions, or damping such as $H_t+\mu I$ globalize the method.

Curvature automatically corrects for scale. Along a high-curvature direction, $H_t^{-1}$ shortens the step; along a low-curvature direction, it permits a longer move. This benefit is expensive in $d$ dimensions: a dense Hessian requires $O(d^2)$ storage and a generic factorization costs $O(d^3)$. Exact Newton methods are therefore compelling for moderate-dimensional smooth problems, but generally impractical for models with millions of parameters.

**Quasi-Newton methods** avoid exact second derivatives. BFGS updates an approximation to the Hessian or inverse Hessian so that it satisfies a secant condition based on successive parameter and gradient differences. L-BFGS stores only a limited history of these difference vectors, reducing memory from quadratic to approximately $O(md)$ for history size $m$. These methods are strong defaults for smooth, deterministic, medium-scale objectives, but noisy mini-batch gradients can corrupt their curvature estimates.

<details>
<summary><strong>Python example: gradient descent, Newton, and BFGS for regularized logistic regression</strong></summary>

```python
import numpy as np
from scipy.optimize import minimize

rng = np.random.default_rng(23)
n_samples, n_features = 500, 6
X = rng.normal(size=(n_samples, n_features))
X = np.column_stack([np.ones(n_samples), X])  # Unpenalized intercept.
true_weights = np.array([-0.3, 1.1, -1.4, 0.5, 0.0, 0.8, -0.6])
probabilities = 1.0 / (1.0 + np.exp(-(X @ true_weights)))
y = rng.binomial(1, probabilities)
regularization = 0.05
penalty_mask = np.r_[0.0, np.ones(n_features)]

def sigmoid(values):
    # Clipping avoids overflow without changing ordinary values.
    return 1.0 / (1.0 + np.exp(-np.clip(values, -35.0, 35.0)))

def objective(weights):
    scores = X @ weights
    # logaddexp(0, score) - y*score is stable binary NLL.
    data_loss = np.mean(np.logaddexp(0.0, scores) - y * scores)
    penalty = 0.5 * regularization * np.sum(penalty_mask * weights ** 2)
    return data_loss + penalty

def gradient(weights):
    return (
        X.T @ (sigmoid(X @ weights) - y) / n_samples
        + regularization * penalty_mask * weights
    )

def hessian(weights):
    fitted = sigmoid(X @ weights)
    curvature = fitted * (1.0 - fitted)
    return (
        X.T @ (curvature[:, None] * X) / n_samples
        + regularization * np.diag(penalty_mask)
    )

# A conservative first-order baseline.
gd_weights = np.zeros(X.shape[1])
for _ in range(1000):
    gd_weights -= 0.2 * gradient(gd_weights)

# Damped Newton steps with backtracking guarantee accepted descent.
newton_weights = np.zeros(X.shape[1])
newton_iterations = 0
for newton_iterations in range(1, 30):
    direction = np.linalg.solve(hessian(newton_weights), -gradient(newton_weights))
    step = 1.0
    current = objective(newton_weights)
    while objective(newton_weights + step * direction) > current + 1e-4 * step * gradient(newton_weights) @ direction:
        step *= 0.5
    newton_weights += step * direction
    if np.linalg.norm(gradient(newton_weights)) < 1e-8:
        break

bfgs_result = minimize(
    objective,
    np.zeros(X.shape[1]),
    jac=gradient,
    method="BFGS",
    options={"gtol": 1e-8, "maxiter": 300},
)

for name, weights, iterations in [
    ("gradient descent", gd_weights, 1000),
    ("Newton", newton_weights, newton_iterations),
    ("BFGS", bfgs_result.x, bfgs_result.nit),
]:
    print(
        f"{name:16s}",
        f"iterations={iterations:4d}",
        f"objective={objective(weights):.8f}",
        f"gradient norm={np.linalg.norm(gradient(weights)):.2e}",
    )
```

</details>

Iteration counts alone are not a cost comparison: a Newton iteration computes and solves with a Hessian, whereas a gradient step is cheap. Wall-clock time, memory, derivative cost, target accuracy, and opportunity for parallelism determine the useful method.

#### **Coordinate Descent**

Coordinate descent minimizes with respect to one parameter or one block while holding the others fixed. A cyclic schedule visits coordinates in a fixed order; randomized coordinate descent samples them. It is effective when each coordinate update has a cheap closed form and when the design matrix or solution is sparse.

For the Lasso objective

$$
J(w)=\frac{1}{2n}\lVert y-Xw\rVert_2^2+\lambda\lVert w\rVert_1,
$$

define a partial residual that removes every feature except $j$,

$$
r_j=y-\sum_{k\neq j}x_kw_k.
$$

Then the exact coordinate update is

$$
w_j\leftarrow
\frac{S_\lambda(x_j^\top r_j/n)}{x_j^\top x_j/n},
\qquad
S_\lambda(z)=\operatorname{sign}(z)\max(|z|-\lambda,0).
$$

The **soft-thresholding operator** sets a coefficient exactly to zero when its correlation with the partial residual does not exceed $\lambda$. Feature scale matters because the same penalty is applied to every coefficient; standardization makes the comparison meaningful. Efficient implementations update a cached residual rather than recomputing the full sum for every coordinate, screen variables that provably remain zero, and use warm starts along a sequence of $\lambda$ values.

<details>
<summary><strong>Python example: Lasso coordinate descent from first principles</strong></summary>

```python
import numpy as np
from sklearn.linear_model import Lasso

rng = np.random.default_rng(31)
n_samples, n_features = 240, 8
X = rng.normal(size=(n_samples, n_features))
X = (X - X.mean(axis=0)) / X.std(axis=0)
true_weights = np.array([2.0, 0.0, -1.4, 0.0, 0.0, 0.7, 0.0, 0.0])
y = X @ true_weights + rng.normal(scale=0.55, size=n_samples)
y = y - y.mean()
regularization = 0.12

def soft_threshold(value, threshold):
    return np.sign(value) * max(abs(value) - threshold, 0.0)

weights = np.zeros(n_features)
residual = y.copy()  # y - X @ weights; initially weights are zero.

for sweep in range(1, 5001):
    largest_change = 0.0
    for feature in range(n_features):
        # Add back feature j, solve its one-dimensional problem, then remove it.
        residual += X[:, feature] * weights[feature]
        correlation = X[:, feature] @ residual / n_samples
        curvature = X[:, feature] @ X[:, feature] / n_samples
        updated = soft_threshold(correlation, regularization) / curvature
        residual -= X[:, feature] * updated
        largest_change = max(largest_change, abs(updated - weights[feature]))
        weights[feature] = updated
    if largest_change < 1e-10:
        break

library_model = Lasso(
    alpha=regularization,
    fit_intercept=False,
    max_iter=10000,
    tol=1e-10,
).fit(X, y)

print("sweeps:", sweep)
print("from-scratch coefficients:", np.round(weights, 4))
print("scikit-learn coefficients: ", np.round(library_model.coef_, 4))
print("maximum coefficient gap:   ", f"{np.max(np.abs(weights - library_model.coef_)):.2e}")
```

</details>

Coordinate descent may slow when features are strongly correlated because one coordinate repeatedly undoes another's progress. Block updates, active sets, or proximal methods can be preferable. It is also not automatically suitable for a dense neural network merely because that network has many coordinates; the useful closed-form subproblem is the crucial property.

#### **Proximal Gradient Methods**

Many objectives split into a differentiable part $g$ and a convex but possibly non-differentiable part $h$:

$$
F(\theta)=g(\theta)+h(\theta).
$$

Ordinary gradient descent cannot directly differentiate $|\theta_j|$ at zero. Proximal gradient takes a gradient step on $g$ and then applies the **proximal operator** of $h$:

$$
\begin{aligned}
z_t &= \theta_t-\eta\nabla g(\theta_t),\\
\theta_{t+1} &= \operatorname{prox}_{\eta h}(z_t),\\
\operatorname{prox}_{\eta h}(z)
&=\arg\min_u\left\{h(u)+\frac{1}{2\eta}\lVert u-z\rVert_2^2\right\}.
\end{aligned}
$$

The second line is not an arbitrary projection. It finds a compromise between reducing the structured penalty and remaining near the gradient proposal $z_t$. For $h(w)=\lambda\lVert w\rVert_1$, the proximal operator is element-wise soft thresholding $S_{\eta\lambda}$; the resulting algorithm is ISTA. If $\nabla g$ is $L$-Lipschitz, $\eta\leq1/L$ gives a standard convergence guarantee for convex $F$. FISTA adds an extrapolation step and improves the worst-case objective rate from $O(1/t)$ to $O(1/t^2)$ under the same convex setting.

<details>
<summary><strong>Python example: ISTA solves sparse regression without differentiating L1 at zero</strong></summary>

```python
import numpy as np
from sklearn.linear_model import Lasso

rng = np.random.default_rng(37)
n_samples, n_features = 300, 12
X = rng.normal(size=(n_samples, n_features))
X = (X - X.mean(axis=0)) / X.std(axis=0)
true_weights = np.array([1.8, 0.0, 0.0, -1.2, 0.0, 0.6, 0.0, 0.0, 0.0, 0.4, 0.0, 0.0])
y = X @ true_weights + rng.normal(scale=0.6, size=n_samples)
y -= y.mean()
regularization = 0.10

def soft_threshold(vector, threshold):
    return np.sign(vector) * np.maximum(np.abs(vector) - threshold, 0.0)

def objective(weights):
    residual = X @ weights - y
    return 0.5 * np.mean(residual ** 2) + regularization * np.sum(np.abs(weights))

# Gradient of the smooth least-squares term has Lipschitz constant ||X||_2^2 / n.
lipschitz_constant = np.linalg.norm(X, ord=2) ** 2 / n_samples
step_size = 1.0 / lipschitz_constant
weights = np.zeros(n_features)

for iteration in range(1, 10001):
    smooth_gradient = X.T @ (X @ weights - y) / n_samples
    updated = soft_threshold(
        weights - step_size * smooth_gradient,
        step_size * regularization,
    )
    if np.linalg.norm(updated - weights, ord=np.inf) < 1e-10:
        weights = updated
        break
    weights = updated

library_model = Lasso(
    alpha=regularization,
    fit_intercept=False,
    max_iter=20000,
    tol=1e-10,
).fit(X, y)

print("iterations:", iteration)
print("ISTA coefficients:        ", np.round(weights, 4))
print("scikit-learn coefficients:", np.round(library_model.coef_, 4))
print("objective:", f"{objective(weights):.6f}")
print("nonzero coefficients:", np.count_nonzero(np.abs(weights) > 1e-8))
```

</details>

Projection is a special proximal operation: the proximal operator of the indicator function of a feasible set is Euclidean projection onto that set. This connection lets the same language describe sparse regularization, bound constraints, group penalties, nuclear norms, and projected gradient methods.


### **Constraints, Lagrangians, and Duality**

Regularization expresses preference by charging a penalty. A constrained problem instead declares some parameter values inadmissible:

$$
\begin{aligned}
\underset{\theta}{\operatorname{minimize}}\quad & f(\theta)\\
\text{subject to}\quad
& g_i(\theta)\leq0,\quad i=1,\ldots,m,\\
& h_j(\theta)=0,\quad j=1,\ldots,p.
\end{aligned}
$$

Constraints arise from norm budgets, non-negativity, probability-simplex requirements, fairness or safety limits, monotonicity, resource budgets, and physical consistency. They change both the set of acceptable solutions and the meaning of optimality: at a boundary, the ordinary gradient need not be zero because every descent direction may point outside the feasible set.

![The Lagrangian connects a constrained primal problem to its dual lower bound, while KKT conditions characterize the balance at an optimum.](assets/kkt-primal-dual.svg){fig-align="center" width="100%" fig-alt="Diagram connecting primal feasibility, the Lagrangian, the dual problem, and the four KKT conditions"}

*The sign convention and optimality conditions follow [Boyd and Vandenberghe, Convex Optimization](https://web.stanford.edu/~boyd/cvxbook/).*

#### **Lagrange Multipliers**

The **Lagrangian** incorporates constraints through multipliers:

$$
\mathcal L(\theta,\lambda,\nu)
=f(\theta)
+\sum_{i=1}^{m}\lambda_i g_i(\theta)
+\sum_{j=1}^{p}\nu_j h_j(\theta),
\qquad \lambda_i\geq0.
$$

$\lambda_i$ is associated with inequality $g_i(\theta)\leq0$, while $\nu_j$ belongs to an equality and can have either sign. Non-negative inequality multipliers preserve lower-bound arguments for a minimization problem: at every feasible $\theta$, $\lambda_i g_i(\theta)\leq0$. Texts that write inequalities in the opposite direction must also reverse the multiplier convention.

Geometrically, consider one active equality $h(\theta)=0$. Feasible infinitesimal directions are tangent to the constraint surface, so no tangent direction can lower the objective at a regular optimum. Therefore the objective gradient must be normal to that surface:

$$
\nabla f(\theta^*)+\nu^*\nabla h(\theta^*)=0.
$$

Economically, a multiplier is a **shadow price**. Under regularity conditions, it approximates how the optimal objective changes if the corresponding constraint bound is relaxed. A large magnitude identifies a constraint that strongly limits the solution; a zero inequality multiplier indicates no first-order value from relaxing an inactive constraint.

<details>
<summary><strong>Python example: projecting onto an equality constraint with a KKT system</strong></summary>

```python
import numpy as np

# Find the point closest to 'target' subject to c^T w = bound.
target = np.array([3.0, -1.0, 2.0])
c = np.array([1.0, 2.0, -1.0])
bound = 0.5

# Stationarity: w - target + multiplier*c = 0
# Feasibility:  c^T w = bound
kkt_matrix = np.block([
    [np.eye(3), c[:, None]],
    [c[None, :], np.zeros((1, 1))],
])
right_hand_side = np.r_[target, bound]
solution = np.linalg.solve(kkt_matrix, right_hand_side)
weights, multiplier = solution[:3], solution[3]

# The same projection also has a short geometric formula.
closed_form = target - ((c @ target - bound) / (c @ c)) * c

print("projected point:", np.round(weights, 5))
print("multiplier:", round(multiplier, 5))
print("constraint residual:", f"{c @ weights - bound:.2e}")
print("stationarity residual:", f"{np.linalg.norm(weights - target + multiplier * c):.2e}")
print("matches geometric projection:", np.allclose(weights, closed_form))
```

</details>

This block linear system is the simplest example of a KKT system. Large constrained solvers exploit its sparsity and structure rather than treating it as an arbitrary dense matrix.

#### **Primal and Dual Problems**

The original constrained optimization is the **primal problem**. The **dual function** minimizes the Lagrangian over the primal parameter while holding the multipliers fixed:

$$
q(\lambda,\nu)=\inf_\theta \mathcal L(\theta,\lambda,\nu).
$$

For every $\lambda\geq0$, $q(\lambda,\nu)$ is a lower bound on the objective value of every primal-feasible point. The **Lagrange dual problem** searches for the tightest such bound:

$$
\underset{\lambda,\nu}{\operatorname{maximize}}\quad q(\lambda,\nu)
\qquad\text{subject to}\qquad \lambda\geq0.
$$

Let $p^*$ and $d^*$ be the optimal primal and dual values. **Weak duality** always gives $d^*\leq p^*$ for this minimization convention. The difference $p^*-d^*$ is the optimal duality gap. For a convex primal problem satisfying an appropriate constraint qualification, such as Slater's condition for convex inequalities, **strong duality** gives $d^*=p^*$. A feasible primal-dual pair with a small gap is then a computable certificate of near-optimality.

Duality is useful beyond proof. A dual problem can decompose over data or resources, expose sensitivity through multipliers, yield screening rules, or be easier to optimize than the primal. Strong duality is not automatic for non-convex problems; a dual lower bound may remain informative while leaving a nonzero gap.

<details>
<summary><strong>Python example: primal and dual values meet at the constrained optimum</strong></summary>

```python
import numpy as np

# Primal: minimize 0.5*(w - 2)^2 subject to w <= 1.
def primal_objective(w):
    return 0.5 * (w - 2.0) ** 2

# L(w, lambda) = 0.5*(w - 2)^2 + lambda*(w - 1).
# Minimizing L over w gives w(lambda)=2-lambda and q=lambda-0.5*lambda^2.
def dual_function(multiplier):
    return multiplier - 0.5 * multiplier ** 2

primal_grid = np.linspace(-1.0, 1.0, 2001)
dual_grid = np.linspace(0.0, 3.0, 3001)
primal_solution = primal_grid[np.argmin(primal_objective(primal_grid))]
dual_multiplier = dual_grid[np.argmax(dual_function(dual_grid))]

primal_value = primal_objective(primal_solution)
dual_value = dual_function(dual_multiplier)
print("primal solution and value:", primal_solution, round(primal_value, 6))
print("dual multiplier and value:", dual_multiplier, round(dual_value, 6))
print("duality gap:", f"{primal_value - dual_value:.2e}")
```

</details>

The unconstrained minimizer is $w=2$, but it is infeasible. The active boundary forces $w^*=1$; the optimal multiplier $\lambda^*=1$ exactly balances the objective gradient at that boundary.

#### **KKT Conditions**

The **Karush-Kuhn-Tucker (KKT) conditions** combine feasibility and first-order balance. For a candidate $(\theta^*,\lambda^*,\nu^*)$, they are:

$$
\begin{array}{ll}
\text{primal feasibility:}
&g_i(\theta^*)\leq0,\quad h_j(\theta^*)=0,\\[2mm]
\text{dual feasibility:}
&\lambda_i^*\geq0,\\[2mm]
\text{stationarity:}
&\nabla f(\theta^*)
+\sum_i\lambda_i^*\nabla g_i(\theta^*)
+\sum_j\nu_j^*\nabla h_j(\theta^*)=0,\\[2mm]
\text{complementary slackness:}
&\lambda_i^*g_i(\theta^*)=0\quad\text{for every }i.
\end{array}
$$

Complementary slackness expresses an either-or relationship. If $g_i(\theta^*)<0$, the constraint has slack and its multiplier must be zero. If $\lambda_i^*>0$, that constraint must be active at equality. An active constraint can still have a zero multiplier in a degenerate problem, so “active” alone does not imply that relaxing it has first-order value.

For differentiable convex objectives and constraints with strong duality, KKT conditions are sufficient for global optimality and, under standard qualifications, necessary. For a non-convex problem they are generally only necessary local conditions: a KKT point can be a local minimum, maximum, saddle, or otherwise non-global point. Constraint qualifications also matter because pathological constraint representations can invalidate multiplier-based necessity.

<details>
<summary><strong>Python example: checking all KKT residuals, including an inactive constraint</strong></summary>

```python
import numpy as np

# Minimize 0.5*(w-2)^2 with g1(w)=w-1<=0 and g2(w)=-w-3<=0.
# The first bound is active at w*=1; the lower bound w>=-3 is inactive.
w_star = 1.0
multipliers = np.array([1.0, 0.0])
constraint_values = np.array([w_star - 1.0, -w_star - 3.0])
constraint_gradients = np.array([1.0, -1.0])
objective_gradient = w_star - 2.0

stationarity = objective_gradient + multipliers @ constraint_gradients
complementarity = multipliers * constraint_values

print("primal feasible:", np.all(constraint_values <= 1e-12))
print("dual feasible:  ", np.all(multipliers >= -1e-12))
print("stationarity residual:", f"{abs(stationarity):.2e}")
print("complementarity residuals:", complementarity)
print("inactive lower-bound multiplier:", multipliers[1])
```

</details>

In numerical optimization, these conditions become residuals rather than exact equalities. A solver may stop only when primal infeasibility, dual infeasibility, stationarity error, and duality gap are all below tolerances appropriate to the scale of the problem.


### **Convergence and Stopping Criteria**

An optimization trace can become flat for several different reasons: it has approached a stationary point, the learning rate is too small, gradients have poor numerical scale, stochastic noise masks progress, or the implementation is wrong. “The loss stopped changing” is therefore an observation, not a convergence proof. Reliable training monitors quantities tied to the mathematical problem as well as held-out behavior and computational budget.

![Learning rate, conditioning, optimization residuals, validation behavior, and resource budgets jointly determine when training should stop.](assets/optimization-stopping.svg){fig-align="center" width="100%" fig-alt="Diagram showing stable and unstable learning rates, ill conditioning, and several optimization and validation stopping signals"}

#### **Learning Rates and Conditioning**

A differentiable function is **$L$-smooth** if its gradient does not change too abruptly:

$$
\lVert\nabla J(u)-\nabla J(v)\rVert_2
\leq L\lVert u-v\rVert_2.
$$

$L$ is an upper curvature scale. For a quadratic $J(\theta)=\tfrac12\theta^\top A\theta$ with positive-definite $A$, the Hessian is $A$ and $L=\lambda_{\max}(A)$. Decompose the error into eigenvector directions of $A$. Gradient descent multiplies the component associated with eigenvalue $\lambda_i$ by

$$
1-\eta\lambda_i
$$

at every iteration. Stability requires $|1-\eta\lambda_i|<1$ for every direction, hence $0<\eta<2/L$. A step at most $1/L$ is a common monotone-descent choice for this convex quadratic. A larger step can alternate across a valley; beyond $2/L$, at least one component grows instead of decaying.

If the objective is also $\mu$-strongly convex, $\mu$ is a lower curvature scale and

$$
\kappa=\frac{L}{\mu}
$$

is the **condition number**. Large $\kappa$ means that one learning rate must accommodate both steep and flat directions. With step $1/L$, the slow direction contracts only by about $1-1/\kappa$ per iteration. Feature standardization, whitening, normalization, momentum, Newton curvature, and adaptive diagonal scaling can all be understood as attempts to improve the effective conditioning.

<details>
<summary><strong>Python example: stable, slow, and divergent learning rates</strong></summary>

```python
import numpy as np

A = np.diag([1.0, 100.0])
start = np.array([10.0, 10.0])

def objective(point):
    return 0.5 * point @ A @ point

def gradient_descent(learning_rate, steps=60):
    point = start.copy()
    for _ in range(steps):
        point -= learning_rate * (A @ point)
    return point

for learning_rate in [0.005, 0.019, 0.021]:
    point = gradient_descent(learning_rate)
    print(
        f"learning rate={learning_rate:.3f}",
        "point =", np.round(point, 4),
        "objective =", f"{objective(point):.4e}",
    )

# Exact inverse-curvature preconditioning makes both directions equally scaled.
preconditioned = start - np.linalg.solve(A, A @ start)
print("one preconditioned step:", preconditioned)
```

</details>

The theoretical range is a guide under its assumptions, not a universal learning-rate calculator. Neural objectives change curvature as parameters move; mini-batch gradients are noisy; and adaptive methods use coordinate-specific effective rates. Warm-up can prevent unstable early updates, decay can reduce late-stage noise, and line search can choose a step from actual objective evaluations. Schedules do not repair unscaled inputs, exploding activations, an incorrect loss reduction, or corrupted gradients.

#### **Gradient, Objective, and Validation-Based Stopping**

Different stopping rules answer different questions:

- **Gradient norm:** $\lVert\nabla J(\theta_t)\rVert\leq\varepsilon_g$ asks whether an unconstrained smooth problem is near first-order stationarity. The tolerance must reflect parameter and objective scale. A tiny gradient can also occur on a plateau far from a useful solution.
- **Relative objective change:** $|J_t-J_{t-1}|/\max(1,|J_{t-1}|)\leq\varepsilon_f$ detects diminishing numerical improvement. It can fire when the learning rate is too small even though the gradient is substantial.
- **Step size:** $\lVert\theta_t-\theta_{t-1}\rVert/\max(1,\lVert\theta_{t-1}\rVert)\leq\varepsilon_\theta$ checks parameter motion, but adaptive scaling or tiny learning rates can make the step misleadingly small.
- **Constrained residuals:** primal feasibility, stationarity, complementary slackness, and a primal-dual gap are more meaningful than an unconstrained gradient norm at a boundary.
- **Validation-based stopping:** a monitored held-out loss or metric has not improved by at least `min_delta` for `patience` checks. This chooses a training duration for generalization; it does not establish that the training objective has converged.
- **Resource limits:** maximum epochs, evaluations, elapsed time, energy, or monetary budget provide a necessary guardrail even when mathematical tolerances are unmet.

For stochastic optimization, one mini-batch gradient or loss is too noisy for a reliable test. Use epoch averages, an exponential moving average, a larger diagnostic batch, or confidence intervals. Validate at a fixed cadence, preserve the best parameters rather than the last parameters, and reserve the test set for the final assessment. Repeatedly adapting decisions to one validation set can itself overfit that set.

<details>
<summary><strong>Python example: validation patience can stop before gradient convergence</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(43)
n_train, n_validation, n_features = 80, 1000, 120
X_train = rng.normal(size=(n_train, n_features))
X_validation = rng.normal(size=(n_validation, n_features))
true_weights = np.zeros(n_features)
true_weights[:6] = np.array([1.8, -1.5, 1.2, 0.9, -0.7, 0.5])

# The underdetermined training set contains enough noise to reward eventual overfit.
y_train = X_train @ true_weights + rng.normal(scale=2.5, size=n_train)
y_validation = X_validation @ true_weights + rng.normal(
    scale=2.5, size=n_validation
)

def mean_squared_error(X, y, weights):
    return np.mean((X @ weights - y) ** 2)

lipschitz_constant = np.linalg.norm(X_train, ord=2) ** 2 / n_train
learning_rate = 0.8 / lipschitz_constant
weights = np.zeros(n_features)
best_weights = weights.copy()
best_validation = float("inf")
best_iteration = 0
patience = 250
min_delta = 1e-5

for iteration in range(1, 10001):
    gradient = X_train.T @ (X_train @ weights - y_train) / n_train
    weights -= learning_rate * gradient
    validation_loss = mean_squared_error(X_validation, y_validation, weights)

    if validation_loss < best_validation - min_delta:
        best_validation = validation_loss
        best_iteration = iteration
        best_weights = weights.copy()
    elif iteration - best_iteration >= patience:
        break

final_gradient = X_train.T @ (X_train @ weights - y_train) / n_train
print("best validation iteration:", best_iteration)
print("stopped at iteration:      ", iteration)
print("training MSE at best:      ", f"{mean_squared_error(X_train, y_train, best_weights):.4f}")
print("validation MSE at best:    ", f"{best_validation:.4f}")
print("validation MSE when stopped:", f"{validation_loss:.4f}")
print("training-gradient norm when stopped:", f"{np.linalg.norm(final_gradient):.3e}")
```

</details>

The patience window protects against reacting to one unlucky validation check. Its value should be interpreted in checks, not blindly in epochs, and should be long enough to span ordinary metric noise and learning-rate schedule transitions. After stopping, restore the best checkpoint and record both the chosen iteration and the rule that chose it.

### **Choosing an Optimization Strategy**

Optimizer selection starts from the mathematical structure and scale of the problem, not from a universal ranking. The following table gives defensible starting points rather than immutable prescriptions.

| Problem structure | Useful starting method | Why it fits | Main caution |
|---|---|---|---|
| Smooth convex objective, moderate dimension, deterministic gradients | L-BFGS or line-search gradient method | Strong progress at high accuracy without a dense Hessian | Full gradients and line searches can be costly on very large data |
| Twice-differentiable objective with modest dimension | Damped Newton or Newton-CG | Curvature handles scaling and gives fast local convergence | Hessian construction, products, or solves may dominate cost |
| Very large, streaming, or frequently updated dataset | SGD or mini-batch SGD | Cheap updates and constant-size working batches | Requires schedules and noise-aware stopping |
| Large non-convex neural model | AdamW for a robust start; momentum SGD as a tuned alternative | Adaptive scaling or momentum handles noisy, anisotropic gradients | Optimizer state, learning-rate schedule, and final generalization must be compared |
| Least squares or generalized linear model with separable L1 penalty | Coordinate descent | Closed-form coordinate updates exploit sparsity | Correlated features can cause slow cycling |
| Smooth loss plus a structured non-smooth penalty | Proximal gradient, FISTA, or a primal-dual method | Preserves exact structure such as sparsity or group selection | Needs a tractable proximal operator and suitable step size |
| Smooth objective with a simple convex feasible set | Projected gradient | Each update restores feasibility by projection | Projection may itself be expensive for a complicated set |
| General nonlinear constraints at moderate scale | Sequential quadratic programming, interior-point, or augmented-Lagrangian method | Directly controls feasibility and optimality residuals | Scaling and feasible initialization can be decisive |

A practical decision sequence is:

1. **Classify the objective.** Is it smooth, non-smooth but composite, constrained, convex, or non-convex? An L1 term should trigger a proximal or coordinate-aware method rather than an undefined gradient at zero.
2. **Estimate scale.** Record the number of examples, parameters, nonzero entries, and accelerator memory. Exact curvature may be sensible at $d=50$ and impossible at $d=10^8$.
3. **Diagnose conditioning.** Standardize features and inspect gradient scales before adding optimizer complexity. Poor coordinates can make every method look unstable.
4. **Choose the gradient regime.** Full gradients favor deterministic line search and quasi-Newton methods; noisy mini-batches favor momentum or adaptive first-order updates.
5. **Define the required accuracy.** A rough predictive model may need only a few passes, while scientific estimation or a tight constrained solution may require small residuals and a duality-gap certificate.
6. **Tune methods fairly.** Give each optimizer an appropriate learning-rate range and schedule under the same data split, preprocessing, initialization policy, and compute budget.
7. **Monitor both optimization and generalization.** Log training objective, gradient or KKT residuals where available, validation behavior, update count, examples processed, wall time, and peak memory.

The optimizer can also create an **implicit bias**: among many interpolating solutions, different update rules, initialization scales, or stopping times may prefer different parameter norms and functions even without an explicit penalty. Consequently, “same architecture and final training loss” does not guarantee the same learned predictor.

The chapter's central distinction is now complete. The objective defines what fitting means; regularization or constraints encode preference; the optimizer exploits the objective's geometry and data scale; stopping decides how much optimization is enough. Chapter 06 evaluates the resulting model on held-out data and separates optimization success from predictive usefulness.

[Back to Machine Learning guideline](Machine Learning.html)
